# From the coursework to a model factory — a CSED 504 mini-capstone
### one training loop, made **fast on every machine**, for **CNNs *and* Transformers**, up to **1.28M images**

Across **501 → 502 → 503** we learned these models *by hand*. Then we spent **months** making them
actually **train** — fast, correctly, on every machine we could get (**Mac / Windows / Linux / Colab**),
and at a scale the coursework never touched (**ImageNet-32: 1.28M images**). This notebook is the receipt.

It runs top to bottom and:
1. rebuilds the training loop we first wrote by hand in 502, then **measures** the tricks (and the traps)
   that made it fast — the *rewrites*, with numbers and graphs;
2. **trains real models** (ResNet-18 and a ViT) with a live **dashboard**;
3. scales up to **ImageNet-32** to show *where the CNN breaks down and the Transformer becomes the win*;
4. **predicts** a big run's cost from small benchmarks, then checks the prediction against reality.

### The ladder: how much of the training do *you* write?
- **501 — classical ML (`sklearn`/`pandas`).** `model.fit()` *is* the training — no loop, no device.
- **502 — drop to the metal.** A hand-written **`Solver`**, ConvNets from scratch, then the PyTorch intro
  and hand-built attention for captioning. **You** write the loop.
- **503 — industrialize it.** Self-attention from scratch (minGPT), the HF `Trainer`, and the first
  cross-platform sweep (`hyper_sweep.py` + `cuda_check.py` — the ancestor of `src/common/gpu_check.py`).

`sklearn.fit()` → write the loop → industrialize it. Two rungs were only *just* reached: **GPU training**
(stops at `.to(device)`; we never wrote mixed precision) and **Vision Transformers** (we did attention for
*text*, never for image patches). This notebook adds the next rung — and shows the scars.

## ⚙️ Configuration — `fast` vs `full`

**Read this cell first.** Everything downstream scales off `MODE`.

- **`'fast'`** — small subsets, a few epochs. The whole notebook runs in a couple of minutes so you can
  *watch it work* and check the pipeline end-to-end. Numbers are illustrative, not final.
- **`'full'`** — the real thing: full CIFAR-10 (~92%), and **full ImageNet-32 (1.28M images)** which takes
  **hours**. Flip to this when you're ready to produce the actual results; the dashboard (below) tracks the
  long runs with a live ETA.

Leave it on `'fast'` to explore; switch to `'full'` for the capstone run.

In [ ]:
# The one knob for the whole notebook. Every stage downstream reads its sizes from MODE, so the
# choice between a quick illustration run and the real multi-hour run is this single edit. In 'fast'
# we shrink both the dataset (a random subset) and the epoch count; in 'full' we use everything.
MODE = 'fast'            # 'fast' = minutes, watch it work   |   'full' = real results, hours

CONFIG = {
    'fast': dict(
        cifar_epochs   = 3,        # CIFAR-10/100: epochs
        cifar_subset   = 8_000,    #   ...on this many training images (None = all 50k)
        in32_epochs    = 2,        # ImageNet-32: epochs
        in32_subset    = 30_000,   #   ...on this many of the 1.28M images (a pipeline check, not the crossover)
    ),
    'full': dict(
        cifar_epochs   = 24,       # ~92% ResNet-18 on CIFAR-10
        cifar_subset   = None,     #   all 50k
        in32_epochs    = 40,       # the real ImageNet-32 schedule
        in32_subset    = None,     #   all 1,281,167 images  (this is the multi-hour run)
    ),
}[MODE]

print(f"MODE = {MODE!r}")
for k, v in CONFIG.items():
    print(f"  {k:14s} = {v}")
if MODE == 'fast':
    print("\n  -> pipeline/illustration run. Flip MODE='full' for real accuracy + the true crossover.")

## 1. From `sklearn.fit()` to a `Solver` to the loop

In **501** you called `model.fit()` and never saw the loop. In **502** you *wrote* it — `Solver._step()`
by hand. PyTorch is that same loop once more, and it's the one piece PyTorch still makes you write — so
it's where every "make it fast" trick below has to live. (`cifar10_train.ipynb` walks the full mapping.)

| CSED 501 | CSED 502, by hand | PyTorch (here) |
|---|---|---|
| `sklearn` did it | `softmax_loss` (`layers.py`) | `nn.CrossEntropyLoss` |
| `sklearn` did it | `ThreeLayerConvNet` (`cnn.py`) | `torchvision.models.resnet18` |
| `model.fit()` | `Solver._step()` (`solver.py`) | the `for x, y in loader:` loop |
| `sklearn` did it | `Solver._save_checkpoint()` | `torch.save(model.state_dict())` |

In [ ]:
# Step 1: Put the repo's sibling folders on the import path.
#
# We do not want to copy-paste the shared machinery into this notebook; we want to import the exact
# same code the offline fleet runs, so that a fix in one place fixes both. The three folders we borrow
# from are ../common (gpu_check.py, the device detection that grew out of the cuda_check.py we wrote in
# 503), the local a1-cv folder (models.py plus cifar_data.py and imagenet_data.py, the GPU-resident
# loaders), and perf/ (perfkit.py, the training-cost estimator we use in section 8).
import os, sys, time, math
for rel in ('../common', '.', 'perf'):
    p = os.path.normpath(os.path.join(os.getcwd(), rel))
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)                       # insert(0) so our code wins over any same-named pip package.


# Step 2: The usual imports. Each inline note says what the import is for here, so no line is a mystery.
import numpy as np                      # Arrays plus the RNG we seed for reproducibility.
import torch                            # Tensors, autograd, and the GPU: the whole engine.
import torch.nn as nn                   # Layers and losses (nn.CrossEntropyLoss lives here).
import matplotlib.pyplot as plt         # Every graph below is one of these.
from tqdm.auto import tqdm              # The within-epoch progress bar (half of our live dashboard).

from gpu_check import get_device, set_seed, enable_fast_matmul
import models as M                      # M.build('resnet18'|'vit'|'resnet50'|'vit_base', num_classes).


# Step 3: Seed torch, numpy, and Python's random so two runs of this notebook agree. Reproducibility
# is a review tool: if a number moves between runs, it is the code that changed, not luck, and that is
# exactly what you want when you are grading a solution.
set_seed(42)

## 2. One device, four platforms

502's GPU cell was three lines and broke the instant a teammate opened it on a **Mac** (Apple **MPS**,
not CUDA), on **Colab** (a different GPU each session), or on a **CPU** laptop. `get_device()` is what
`csed503/cuda_check.py` grew into: it prefers **CUDA → MPS → CPU**, *smoke-tests* MPS (some macOS builds
advertise it and then crash on the first tensor op), and on a multi-GPU box picks the best same-architecture
set. Same notebook, four platforms.

In [ ]:
# Step 1: Ask gpu_check for the best device on this machine. This one call is our "runs on four
# platforms" story: it returns CUDA here, Apple MPS on a Mac, or the CPU, and on our two-card box it
# also picks the best same-architecture GPU set.
DEVICE = get_device()


# Step 2: On CUDA only, turn on the free speed switches: TF32 matmuls and the cuDNN autotuner, which
# benchmarks conv algorithms once and then reuses the winner. It is harmless off CUDA, so we call it
# without a guard beyond the device check.
if DEVICE.type == 'cuda':
    enable_fast_matmul()

def pick_amp(device):
    """
    Choose which 16-bit dtype autocast should use, and whether we need a gradient loss-scaler.

    Mixed precision runs the matmuls in 16-bit for speed, but the two 16-bit formats behave
    differently, and picking the wrong one either crashes or quietly trains worse.

     - bf16 keeps fp32's exponent range, so gradients do not underflow and no loss scaler is needed.
       Blackwell GPUs and Zen4 CPUs have it; this is what we want whenever we can get it.
     - fp16 has a tiny exponent range, so small gradients round to zero. The fix is a GradScaler:
       multiply the loss up before backward, then divide the gradients back down before the step.
       Older CUDA cards (a Colab T4) only do fp16, so we have to handle this path too.
     - MPS autocast is still patchy, so on a Mac we skip AMP entirely and train in plain fp32.

    Returns (amp_dtype_or_None, use_scaler), where None means "no autocast, run fp32".
    """
    if device.type == 'cuda':
        return (torch.bfloat16, False) if torch.cuda.is_bf16_supported() else (torch.float16, True)
    if device.type == 'cpu':
        return torch.bfloat16, False    # Zen4 has AVX-512-BF16; harmless (just not faster) on older CPUs.
    return None, False                  # MPS falls back to plain fp32.


# Step 3: Lock the choice in once, up top, and print it so every run announces its own precision.
AMP_DTYPE, USE_SCALER = pick_amp(DEVICE)
print(f'Device {DEVICE} | AMP dtype {AMP_DTYPE} | loss scaler {USE_SCALER}')

## 3. The first wall — *feeding* the GPU

502 handed data to the model with a `DataLoader` and CPU worker processes. Right tool for big images —
it broke us twice at 32×32:

- **Spawn.** On Windows/macOS, DataLoader workers are *spawned* (fresh Python processes) and must
  **re-import** the `Dataset` class. A class defined in a notebook cell lives in `__main__` and can't be
  re-imported — the workers die with `Can't get attribute '...'`. So multi-worker loading was
  Linux/Colab-only; everyone else fell back to `num_workers=0` = **one CPU core** feeding the GPU.
- **The GPU was faster than one core could feed it.** At 32×32 a ResNet-18 trains faster than CPU
  augmentation produces batches, so the card sat *waiting on the CPU*.

The rewrite: **stop using the CPU for data.** CIFAR-10 is ~180 MB — hold it all **on the GPU** and
crop/flip/normalize there. Let's measure the difference on *this* machine.

In [ ]:
from torchvision.datasets import CIFAR10
from cifar_data import GPUImageLoader         # Our on-device loader; see src/a1-cv/cifar_data.py.

# MEAN and STD are the published CIFAR-10 per-channel statistics we normalize with. torchvision hands
# us images as (N, 32, 32, 3) uint8 (rows, cols, then channels last); PyTorch convolutions want
# (N, 3, 32, 32), with channels before the spatial dims, which is the reorder to_device does below.
# Each dataset gets its own subfolder under data/ so CIFAR-10, CIFAR-100, and ImageNet-32 never mix.
MEAN, STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
_root = os.path.join(os.getcwd(), 'data')
_tr = CIFAR10(os.path.join(_root, 'cifar10'), train=True,  download=True)
_te = CIFAR10(os.path.join(_root, 'cifar10'), train=False, download=True)

def to_device(ds):
    """
    Move a torchvision dataset's images and labels onto the GPU, converting the images into the NCHW
    layout that PyTorch convolutions expect.

    Inputs:
     - ds: a torchvision CIFAR dataset, with ds.data of shape (N, 32, 32, 3) uint8

    Returns:
     - x: image tensor on the GPU, of shape (N, 3, 32, 32), uint8
     - y: label tensor on the GPU, of shape (N,)
    """
    # Step 1: Copy the raw image bytes onto the GPU.
    #
    # ds.data: (N, 32, 32, 3)
    # x:       (N, 32, 32, 3)
    #
    # We keep the data as uint8. It is one byte per pixel (about 150 MB for all of CIFAR-10), and
    # converting to float is nearly free per batch on the GPU, so there is no reason to store it as
    # float and pay four times the VRAM.

    x = torch.from_numpy(ds.data).to(DEVICE)


    # Step 2: Reorder the axes from NHWC to NCHW.
    #
    # x (before): (N, 32, 32, 3)
    # x (after):  (N, 3, 32, 32)
    #
    # The permute call only relabels the strides, so no memory actually moves. We call contiguous
    # afterwards to lay the bytes out as NCHW; otherwise the first convolution pays for that copy itself.

    x = x.permute(0, 3, 1, 2).contiguous()


    # Step 3: Move the labels onto the GPU alongside the images.
    #
    # ds.targets: list of length N
    # y:          (N,)

    y = torch.tensor(ds.targets, device=DEVICE)
    return x, y

xtr, ytr = to_device(_tr)                             # The entire training set now lives in VRAM.
xte, yte = to_device(_te)
print(f'train {tuple(xtr.shape)}  test {tuple(xte.shape)}  — all on {DEVICE}')

In [ ]:
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class _CPUAug(Dataset):
    """
    The ordinary way, here only for the comparison: a torchvision Dataset that augments one PIL image
    at a time on the CPU. This is exactly what 502 taught, and exactly what starves a fast GPU at 32x32.
    """
    def __init__(self, imgs, labels):
        self.imgs, self.labels = imgs, labels
        # The same crop and flip we do in one shot on the GPU, but here as per-image PIL calls in Python.
        self.tf = transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD)])
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return self.tf(Image.fromarray(self.imgs[i])), int(self.labels[i])

def feed_rate(loader, n_batches=20):
    """
    Measure how many images per second this loader can hand the model. This is forward-only on
    purpose: we are timing data delivery, not the model, so we keep the backward pass out of the number.
    """
    # Step 1: A throwaway model in eval mode, just to consume batches.
    model = M.build('resnet18', 10).to(DEVICE).eval()
    seen, it = 0, iter(loader)


    # Step 2: Synchronize before starting the clock. On CUDA the kernels launch asynchronously, so
    # without a synchronize() we would be timing how fast Python queues work, not how fast the GPU does it.
    if DEVICE.type == 'cuda': torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        for _ in range(n_batches):
            try:    x, y = next(it)
            except StopIteration: it = iter(loader); x, y = next(it)   # Wrap around if the loader runs dry.
            model(x.to(DEVICE, non_blocking=True)); seen += x.size(0)
    if DEVICE.type == 'cuda': torch.cuda.synchronize()     # Sync again to wait for the GPU to actually finish.
    return seen / (time.time() - t0)

# Step 3: Run the race and plot it. The CPU loader uses num_workers=0, because the spawn problem
# described above is what forced everyone off Linux into exactly this single-core fallback, so this is
# the fair baseline.
cpu_loader = DataLoader(_CPUAug(_tr.data, _tr.targets), batch_size=512, shuffle=True, num_workers=0)
gpu_loader = GPUImageLoader(xtr, ytr, 512, MEAN, STD, train=True, seed=42)
rates = {'CPU DataLoader (workers=0)': feed_rate(cpu_loader),
         'GPU-resident (on-device)':   feed_rate(gpu_loader)}
for k, v in rates.items(): print(f'{k:28s} {v:9,.0f} img/s')

fig, ax = plt.subplots(figsize=(5.5, 3.4))                  # The gap you are about to see is large.
ax.bar(list(rates), list(rates.values()), color=['#c44', '#4a8'])
ax.set_ylabel('images / sec delivered'); ax.set_title('Feeding the GPU at 32×32')
ax.bar_label(ax.containers[0], fmt='%.0f'); plt.xticks(rotation=8); plt.tight_layout(); plt.show()

## 4. Where PyTorch fought back — flips and non-contiguous views

Moving augmentation onto the GPU sounds simple. It wasn't. Two rewrites we didn't see coming.

### (a) The horizontal flip that stalled the pipeline
The obvious way to flip a random half of a batch is a boolean-mask write, `x[flip] = x[flip].flip(-1)`.
But `x[flip]` is *advanced indexing*: PyTorch calls `nonzero()` on the mask to find the selected rows,
and `nonzero()` is a **device→host sync** — the GPU stops and reports *how many* rows matched. Two syncs
per batch, ~2,500 batches/epoch, and it silently **breaks CUDA-graph capture**. The branch-free,
bitwise-identical fix is `torch.where`. Let's time both.

In [ ]:
# B is the batch size (512); C, H, W are 3, 32, 32; flip is a per-image True/False mask of shape (B,).
def time_flip(fn, iters=400):
    """
    Measure flips per second for one strategy, timed in isolation with a synchronize on each side of
    the loop. The reason is the same as in section 3: on CUDA we have to wait for the GPU, or we would
    be timing Python's queue instead of the work.
    """
    x    = torch.randn(512, 3, 32, 32, device=DEVICE)      # (B, C, H, W), a batch of the right shape.
    flip = torch.rand(512, device=DEVICE) < 0.5            # (B,) bool, mirror roughly half the batch.
    if DEVICE.type == 'cuda': torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(iters): fn(x, flip)
    if DEVICE.type == 'cuda': torch.cuda.synchronize()
    return iters / (time.time() - t0)


# Strategy A: the obvious boolean-mask write. This is the trap.
def mask_write(x, flip):
    # x2[flip] is advanced indexing. To know which rows to touch, PyTorch runs nonzero() on the mask,
    # and nonzero()'s output length is data-dependent, so it has to copy that count back to the CPU.
    # That is a device-to-host sync (about two per batch), and it quietly breaks CUDA-graph capture too.
    x2 = x.clone()                                         # Clone so repeated timed calls start clean.
    x2[flip] = x2[flip].flip(-1)                           # Selected rows reversed along W, via a host sync.
    return x2


# Strategy B: branch-free, bitwise-identical, and sync-free.
def where_flip(x, flip):
    # torch.where(cond, a, b) is a pure element-wise select: no nonzero(), no host sync, graph-safe.
    # flip is (B,), so we view it as (B, 1, 1, 1) to broadcast over the image: (B, 1, 1, 1) against
    # (B, C, H, W) gives (B, C, H, W).
    return torch.where(flip.view(-1, 1, 1, 1), x.flip(-1), x)

a, b = time_flip(mask_write), time_flip(where_flip)
print(f'x[flip]=x[flip].flip(-1) : {a:8,.0f} flips/s   (nonzero -> host sync, breaks CUDA graphs)')
print(f'torch.where(...)         : {b:8,.0f} flips/s   ({b/a:.1f}x, and graph-safe)')

### (b) `.view()` on a non-contiguous tensor — the Mac-only crash
The per-sample random crop gathers pixels with fancy indexing, which returns a **non-contiguous** tensor
(its memory layout no longer matches its shape). CUDA usually tolerates that; **Apple MPS** does not — the
ViT's backward pass calls `.view()` on exactly such a tensor and **throws**. Two rules came out of that
afternoon, and they're now baked into the code:

- Use **`.reshape()`** (or an explicit `.contiguous()` first), never a bare **`.view()`**, on anything
  that's been transposed / indexed / permuted. `.view()` *requires* contiguity; `.reshape()` copies if it
  must.
- "Runs on every platform" is something you **debug into existence**, not a checkbox. Our crop returns a
  contiguous result on purpose; the flip is `torch.where`; nothing downstream assumes a memory layout.

## 5. The counterintuitive one — mixed precision × memory format

Mixed precision (**AMP**) was never in the coursework (503 only got it through `Trainer(fp16=True)`).
`torch.autocast` runs the matmuls in low precision — **but which** low precision, and **which memory
layout**, interact in a way that cost us days.

`channels_last` (NHWC) is cuDNN's preferred conv layout, so the folklore is "always use it." On this
hardware that folklore is **half wrong**, and the sign flips with the dtype. Don't trust us — measure it:

In [ ]:
def step_rate(amp_dtype, channels_last, iters=60, warmup=15):
    """
    Measure full training-step throughput (forward, backward, and optimizer) for one (dtype,
    memory-format) pair. Unlike section 3's feed_rate, this includes the backward pass: here we are
    timing the model, not the data. B is 512 (batch); C, H, W are 3, 32, 32.
    """
    # Step 1: Build the model, and only if we are testing channels_last store its weights in NHWC, the
    # layout cuDNN prefers for convolutions.
    m = M.build('resnet18', 10).to(DEVICE)
    if channels_last:
        m = m.to(memory_format=torch.channels_last)
    opt = torch.optim.SGD(m.parameters(), lr=0.01, momentum=0.9)
    x = torch.randn(512, 3, 32, 32, device=DEVICE)         # (B, C, H, W) synthetic batch; only the shape matters here.
    y = torch.randint(0, 10, (512,), device=DEVICE)
    if channels_last:
        # The activations have to be NHWC too, or every conv silently re-copies them each step and we
        # would measure the copies instead of the layout. Match the model.
        x = x.contiguous(memory_format=torch.channels_last)


    # Step 2: One training step, closed over the fixed batch.
    def step():
        opt.zero_grad(set_to_none=True)
        with torch.autocast(DEVICE.type, dtype=amp_dtype, enabled=amp_dtype is not None):
            nn.functional.cross_entropy(m(x), y).backward()
        opt.step()


    # Step 3: Warm up first, then time. cuDNN's autotuner spends the first few steps benchmarking conv
    # algorithms, and timing those would blame the warmup on the config. Never trust iteration 1.
    for _ in range(warmup): step()
    if DEVICE.type == 'cuda': torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(iters): step()
    if DEVICE.type == 'cuda': torch.cuda.synchronize()
    return 512 * iters / (time.time() - t0)                # img/s = (batch * iters) / seconds.

# Step 4: Sweep the five combinations and plot them. Watch the two channels_last bars in particular;
# they are the whole point (see the note that follows this cell).
configs = [('fp32', None, False), ('fp16', torch.float16, False), ('fp16 + chan_last', torch.float16, True),
           ('bf16', torch.bfloat16, False), ('bf16 + chan_last', torch.bfloat16, True)]
rate = {name: step_rate(dt, cl) for name, dt, cl in configs}
base = rate['fp32']
for name in rate: print(f'{name:18s} {rate[name]:9,.0f} img/s   {rate[name]/base:4.2f}x vs fp32')

fig, ax = plt.subplots(figsize=(7, 3.6))                    # Same flag, opposite sign, decided by dtype.
ax.bar(list(rate), list(rate.values()), color=['#888', '#69c', '#c44', '#48a', '#2a6'])
ax.set_ylabel('img/s (resnet18, bs512)'); ax.set_title('Mixed precision × memory format — same GPU')
ax.bar_label(ax.containers[0], fmt='%.0f'); plt.xticks(rotation=12); plt.tight_layout(); plt.show()

Read the bars. **`channels_last` *helps* under bf16 and *hurts* under fp16** — on the RTX PRO 6000 we
measured bf16+`channels_last` as the fastest config *and* fp16+`channels_last` as roughly **4× slower** than
plain fp16 (a pathological cuDNN NHWC-fp16 path). Same flag, opposite sign, decided by the dtype. There is
no substitute for measuring on the target — which is exactly what pushes us toward a **factory** that
measures for us (§8).

The habit under all of it: before optimizing, know **what the GPU is waiting on** —

| symptom | you are… | the fix |
|---|---|---|
| low GPU util, one CPU core pegged | **CPU-bound** (data) | more workers, or on-device data (§3) |
| low GPU util, low CPU util | **launch-bound** (batch too small) | bigger batch; avoid per-batch host syncs (§4) |
| high GPU util | **compute-bound** | you're done — need a bigger model or a bigger card |

## 6. A real training run — with a live dashboard

Now we wire the fast path into one `train()` used for *every* run below (CIFAR here, ImageNet-32 in §9).
For the long runs (`MODE='full'`, ImageNet-32 = hours) staring at a scrolling log is useless, so `train()`
redraws a **dashboard after every epoch**: the loss and accuracy curves so far, plus a status line with
throughput and an **ETA**. Read the comments — this is `Solver.train()`, one rung up, instrumented.

In [ ]:
from IPython.display import clear_output

@torch.no_grad()                                           # Eval never needs gradients, so skip the graph and save memory.
def evaluate(model, val_batches, channels_last):
    """
    Compute top-1 accuracy over one validation pass. val_batches() hands back a fresh (x, y) iterator
    on each call, so we can sweep the validation set whenever we like. B is the batch; C, H, W are
    3, 32, 32.
    """
    # Step 1: Put BatchNorm and Dropout into inference behavior.
    model.eval()


    # Step 2: Count correct predictions on the GPU. Same lesson as the training loop: a per-batch
    # .item() would sync the GPU to the CPU every batch, so we keep a running GPU scalar and sync once.
    correct = torch.zeros((), device=DEVICE); total = 0
    for x, y in val_batches():
        if channels_last: x = x.contiguous(memory_format=torch.channels_last)   # Match the model's layout.
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            preds = model(x).argmax(1)                     # (B, n_classes) logits to (B,) predicted class via argmax.
        correct += (preds == y).sum(); total += y.size(0)  # Accumulate on the GPU, no sync.


    # Step 3: Hand the model back in train mode, and take the one GPU-to-host sync for the whole pass.
    model.train()
    return (correct / total).item()

def dashboard(title, hist, epoch, epochs, t_start):
    """
    Redraw the progress panel in place after each epoch: a status line (with ETA) and the loss and
    accuracy curves so far. For the hours-long ImageNet run a scrolling wall of tqdm is useless; you
    want to see the curve bending and know when it will finish.
    """
    clear_output(wait=True)                                # Replace the previous frame instead of scrolling.
    elapsed = time.time() - t_start
    eta = elapsed / epoch * (epochs - epoch)               # Linear ETA from the mean epoch time so far.
    def hms(s): return f'{int(s//3600)}h{int(s%3600//60):02d}m{int(s%60):02d}s'
    print(f'[{title}]  epoch {epoch}/{epochs}   '
          f'val top-1 {hist["top1"][-1]:6.2%}   loss {hist["loss"][-1]:.3f}   '
          f'{hist["img_s"][-1]:8,.0f} img/s   elapsed {hms(elapsed)}   ETA {hms(eta)}')
    xs = range(1, epoch + 1)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.4))
    a1.plot(xs, hist['loss'], color='#c44'); a1.set_title('train loss'); a1.set_xlabel('epoch')
    a2.plot(xs, hist['top1'], color='#2a6'); a2.set_title('val top-1'); a2.set_xlabel('epoch')
    a2.set_ylim(0, 1); a2.grid(alpha=.3)
    plt.tight_layout(); plt.show(); plt.close(fig)         # Close the figure, or 40 epochs leak 40 figures.

def train(title, model, train_batches, val_batches, n_train, epochs, *, lr, opt='sgd',
          strong_aug=None, live=True):
    """
    The single training loop used for every run below (CIFAR here, ImageNet-32 in section 9). This is
    Solver.train() from 502, one rung up and instrumented.

    Inputs:
     - train_batches()/val_batches(): each returns a fresh (x, y) iterator for one pass
     - n_train: images per epoch, which feeds the img/s number and the tqdm total
     - strong_aug: the ViT's mixup/CutMix and erasing callback, or None for the CNN

    Returns:
     - a history dict of per-epoch loss, val-top1, and img_s
    """
    # Step 1: Move the model to the device, and per the section 5 measurement switch it to
    # channels_last only on CUDA with bf16, where NHWC is a win. Under fp16 it was a 4x loss, so we
    # deliberately do not.
    model = model.to(DEVICE)
    CHAN_LAST = (DEVICE.type == 'cuda' and AMP_DTYPE is torch.bfloat16)
    if CHAN_LAST:
        model = model.to(memory_format=torch.channels_last)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)        # Label smoothing 0.1, a mild and standard regularizer.

    # Step 2: The optimizer and LR schedule follow the architecture (the 502/503 lesson): SGD with
    # momentum for the CNN, AdamW for the transformer. A ViT trained on the CNN's SGD recipe lands
    # about 15 points low, so the recipe is not a detail. The schedule is the modern default: a linear
    # warmup into cosine decay.
    opt_obj = (torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05) if opt == 'adamw'
               else torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, nesterov=True, weight_decay=5e-4))
    warm_ep = min(5, max(1, epochs - 1))                   # About 5 warmup epochs, but never more than the run itself.
    warm = torch.optim.lr_scheduler.LinearLR(opt_obj, 0.1, total_iters=warm_ep)
    cos  = torch.optim.lr_scheduler.CosineAnnealingLR(opt_obj, T_max=max(1, epochs - warm_ep))
    sched = torch.optim.lr_scheduler.SequentialLR(opt_obj, [warm, cos], [warm_ep])
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_SCALER)        # A no-op unless we are in fp16.

    hist = {'loss': [], 'top1': [], 'img_s': []}
    t_start = time.time()
    steps = math.ceil(n_train / 512)                       # Batches per epoch, which is the tqdm total.
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); seen = 0
        # This is the optimization section 4 was really about, now applied to the loop itself. We
        # accumulate the running loss on the GPU and sync exactly once, after the epoch. Writing
        # loss.item() inside the batch loop forces a GPU-to-host sync every step: the CPU then cannot
        # launch the next batch until the GPU stops and reports the loss, so a small 32x32 model stalls
        # between steps and neither the card nor the CPU is stressed (about 90% drops to about 30%
        # utilization). It is exactly the flip's nonzero() host sync from section 4, just hiding in the
        # metrics line.
        loss_sum = torch.zeros((), device=DEVICE)
        for x, y in tqdm(train_batches(), total=steps, desc=f'{title} ep {ep}/{epochs}', leave=False):
            if CHAN_LAST: x = x.contiguous(memory_format=torch.channels_last)   # Activations match the weights.
            opt_obj.zero_grad(set_to_none=True)            # set_to_none frees the grad tensors, a touch faster.
            with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
                if strong_aug is not None:                 # ViT path: mixup/CutMix blends images into two soft targets.
                    x, ya, yb, lam = strong_aug(x, y)
                    out = model(x); loss = lam * crit(out, ya) + (1 - lam) * crit(out, yb)
                else:                                      # CNN path: plain cross-entropy on the true label.
                    loss = crit(model(x), y)
            # Backward and step. Under fp16 the scaler scales the loss up so small gradients survive,
            # then unscales before stepping; under bf16 scaler.is_enabled() is False and this is a no-op.
            if USE_SCALER: scaler.scale(loss).backward(); scaler.step(opt_obj); scaler.update()
            else:          loss.backward(); opt_obj.step()
            loss_sum += loss.detach() * y.size(0)          # Stays on the GPU, so the CPU never waits mid-epoch.
            seen += y.size(0)
        sched.step()                                       # One LR step per epoch, not per batch.
        if DEVICE.type == 'cuda': torch.cuda.synchronize() # Finish the epoch before we read the clock and metrics.
        hist['loss'].append((loss_sum / seen).item())      # The single GPU-to-host sync for the epoch.
        hist['top1'].append(evaluate(model, val_batches, CHAN_LAST))
        hist['img_s'].append(seen / (time.time() - t0))
        if live: dashboard(title, hist, ep, epochs, t_start)
    return hist

### Train ResNet-18 on CIFAR-10
On the workstation GPU `MODE='full'` reaches the low-90s% in ~2 minutes; `MODE='fast'` just proves the
pipeline. The `GPUImageLoader` yields on-device, augmented batches — so `train_batches()` is just "make a
fresh loader pass."

In [ ]:
# Step 1: Build the on-device loaders. In 'fast' MODE we slice a random subset off the front (the data
# was already shuffled at load time, so xtr[:subset] is a fair sample); in 'full' we use all 50k.
def cifar_loaders(subset):
    xt, yt = (xtr, ytr) if subset is None else (xtr[:subset], ytr[:subset])
    train_loader = GPUImageLoader(xt, yt, 512, MEAN, STD, train=True, seed=42)   # Shuffled and augmented.
    test_loader  = GPUImageLoader(xte, yte, 512, MEAN, STD, train=False)         # Clean, in order.
    return train_loader, test_loader, len(yt)

# Step 2: Train the ResNet. train_batches is lambda: iter(c_train), meaning "hand me a fresh augmented
# pass"; the loader reshuffles and re-augments every time we iterate it.
c_train, c_test, c_n = cifar_loaders(CONFIG['cifar_subset'])
resnet_hist = train('ResNet-18 / CIFAR-10',
                    M.build('resnet18', num_classes=10),
                    train_batches=lambda: iter(c_train),
                    val_batches=lambda: iter(c_test),
                    n_train=c_n, epochs=CONFIG['cifar_epochs'],
                    lr=0.1 * 512 / 256)                     # SGD LR scaled linearly with batch (Goyal et al., a 502 lesson).
print(f'ResNet-18 best val top-1: {max(resnet_hist["top1"]):.2%}')

## 7. The same job, a Transformer — new recipe, same fast loop

A **ViT** is the attention we hand-built in 502/503, now over image *patches*. It ports with one argument
(`num_classes`), but its **training recipe does not**: hand it the ResNet's recipe (SGD, `lr=0.1`) and it
lands in the 50s. The fixes — **AdamW**, LR **warmup**, and **strong augmentation** (mixup/CutMix + random
erasing) — are the transformer's *substitute for the inductive bias it doesn't have*. On 50k images the CNN
still wins; hold that thought until §9.

In [ ]:
# Step 1: The ViT's strong augmentation. We reuse the GPU-side mixup/CutMix and random-erasing from
# imagenet_data.py, the exact same code the offline fleet runs, so there is nothing to reimplement.
from imagenet_data import mixup_cutmix, random_erasing_

def vit_aug(x, y):
    """
    Erase a random patch, then apply mixup/CutMix with another image in the batch.

    Only the ViT gets this. A ViT has no locality prior to restrain it, so on crop and flip alone it
    memorizes the training set. The heavy augmentation is its substitute for the inductive bias the CNN
    gets for free from weight-sharing. The CNN would actually be a bit worse with mixup (see the
    section 7 prose).
    """
    x = random_erasing_(x)                                 # Zero out a random rectangle in about 25% of the batch.
    return mixup_cutmix(x, y)                              # Returns (x, y_a, y_b, lam): blended image and two soft targets.

# Step 2: The same train() call, but with the transformer recipe: AdamW instead of SGD, plus strong_aug.
vit_hist = train('ViT / CIFAR-10',
                 M.build('vit', num_classes=10),
                 train_batches=lambda: iter(c_train), val_batches=lambda: iter(c_test),
                 n_train=c_n, epochs=CONFIG['cifar_epochs'],
                 lr=1e-3, opt='adamw', strong_aug=vit_aug)
print(f'ViT best {max(vit_hist["top1"]):.2%}   vs ResNet-18 {max(resnet_hist["top1"]):.2%}  '
      f'-> the CNN wins on 50k images')

# Step 3: Both curves on one axis. This is the "CNN ahead on small data" picture; hold it until
# section 9, where 25x more data flips the result.
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(range(1, len(resnet_hist['top1'])+1), resnet_hist['top1'], label='ResNet-18 (CNN)', color='#2a6')
ax.plot(range(1, len(vit_hist['top1'])+1),    vit_hist['top1'],    label='ViT (Transformer)', color='#96c')
ax.set_xlabel('epoch'); ax.set_ylabel('val top-1'); ax.set_ylim(0, 1); ax.legend(); ax.grid(alpha=.3)
ax.set_title(f'CNN vs Transformer — CIFAR-10 ({c_n:,} images, MODE={MODE})'); plt.tight_layout(); plt.show()

## 8. Benchmark small, **predict** big — will ImageNet-32 take 1 hour or 1 day?

Before launching a run that might take all day, predict it. The key fact: **ImageNet-32 uses the same
models at the same 32×32 resolution as CIFAR** — so the *per-image* compute cost is (almost) identical.
That means we can **benchmark on CIFAR (seconds) and extrapolate to ImageNet-32's 1.28M images**.

We do it two ways and compare them in §9:
1. **Naive throughput extrapolation** — measure steady-state `img/s` on CIFAR, then `time ≈ images × epochs / img_s`. Intuitive, and *slightly optimistic* (we'll see why).
2. **`perfkit`'s roofline** (`../perf/`, the grown-up `hyper_sweep.py`) — counts each model's FLOPs
   *including the 1000-class head ImageNet needs* and divides by the GPU's measured throughput, returning a
   `[fast, slow]` band. This is the factory's redesign gate: *"minutes or days?"* answered with no run.

In [ ]:
# The naive prediction: measure steady-state img/s on CIFAR, then scale by the image count.
def measure_img_s(model, loader, n_train, warmup_batches=10, measure_batches=40):
    """
    Measure steady-state training throughput (full forward, backward, and step). We skip the first
    warmup_batches, since epoch 1 is cuDNN-autotuning and always slow, then time the next
    measure_batches.
    """
    # Step 1: Use the same fast setup train() uses (channels_last on CUDA with bf16), so the benchmark
    # matches the real run and does not flatter or slander it.
    model = model.to(DEVICE)
    CHAN = (DEVICE.type == 'cuda' and AMP_DTYPE is torch.bfloat16)
    if CHAN: model = model.to(memory_format=torch.channels_last)
    opt = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    crit = nn.CrossEntropyLoss()
    it = iter(loader); seen = 0


    # Step 2: One training step. nonlocal it lets us re-open the loader if it runs dry mid-measurement.
    def one():
        nonlocal it
        try: x, y = next(it)
        except StopIteration: it = iter(loader); x, y = next(it)
        if CHAN: x = x.contiguous(memory_format=torch.channels_last)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            crit(model(x), y).backward()
        opt.step(); return x.size(0)


    # Step 3: Warm up, sync, time the measured window, then sync again: the section 3 and 5 timing
    # discipline.
    for _ in range(warmup_batches): one()
    if DEVICE.type == 'cuda': torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(measure_batches): seen += one()
    if DEVICE.type == 'cuda': torch.cuda.synchronize()
    return seen / (time.time() - t0)

# A second benchmark dataset, CIFAR-100 (same 32x32, but 100 classes). Before we trust a CIFAR
# measurement to predict a third dataset (ImageNet-32, 1000 classes), we check that the per-image cost
# is stable across datasets. If 10 versus 100 classes barely moves img/s, the conv/attention backbone
# dominates and we can extrapolate; if it moved a lot, the classifier head would matter and naive
# extrapolation would be unsafe.
from torchvision.datasets import CIFAR100
_c100 = CIFAR100(os.path.join(_root, 'cifar100'), train=True, download=True)
x100 = torch.from_numpy(_c100.data).to(DEVICE).permute(0, 3, 1, 2).contiguous()   # NHWC to NCHW on the GPU (as in section 3).
y100 = torch.tensor(_c100.targets, device=DEVICE)
if CONFIG['cifar_subset'] is not None:
    x100, y100 = x100[:CONFIG['cifar_subset']], y100[:CONFIG['cifar_subset']]
c100_train = GPUImageLoader(x100, y100, 512, MEAN, STD, train=True, seed=42)   # Its norm constants do not affect timing.

# Benchmark each model on both datasets, average the two (they should agree), then extrapolate:
# predicted hours = (ImageNet images * epochs) / (img/s) / 3600.
IN32_N, IN32_EPOCHS = 1_281_167, CONFIG['in32_epochs']       # ImageNet-32 full size and our epoch budget.
bench = {}
for name in ('resnet18', 'vit'):
    r10  = measure_img_s(M.build(name, 10),  c_train,    c_n)        # Benchmark on CIFAR-10 (10 classes).
    r100 = measure_img_s(M.build(name, 100), c100_train, len(y100))  # Benchmark on CIFAR-100 (100 classes).
    r = 0.5 * (r10 + r100)                                   # About equal, so one stable per-image rate to extrapolate from.
    naive_hours = IN32_N * IN32_EPOCHS / r / 3600            # (images * epochs) / (img/s) to hours.
    bench[name] = dict(img_s=r, naive_hours=naive_hours)
    print(f'{name:9s}: CIFAR-10 {r10:8,.0f}  |  CIFAR-100 {r100:8,.0f} img/s  (consistent)  ->  '
          f'naive ImageNet-32 ({IN32_N:,}×{IN32_EPOCHS} ep): {naive_hours:5.2f} h')

In [ ]:
# The FLOPs-aware roofline (perfkit), which counts the ImageNet 1000-class head.
import perfkit as pk

# Step 1: Probe the GPU's real peak throughput with a quick burst-GEMM (a few seconds, not a full run).
# Take bf16 if the card has it, else tf32, else fp32: the fastest number this hardware can actually hit.
peak = pk.probe_matmul_tflops(DEVICE)
p_tflops = peak.get('bf16') or peak.get('tf32') or peak.get('fp32')
print(f'GPU peak ~{p_tflops:.0f} TFLOPS\n')

# Step 2: For each model, count its FLOPs per image with num_classes=1000. This is exactly why the
# roofline beats the naive number: it sees the ImageNet classifier head that our 10-class CIFAR
# benchmark never charged us for. tier1_estimate then returns an optimistic-to-pessimistic hours band
# over a range of realistic GPU efficiencies (MFU), the honest "minutes or days?" bracket, with no
# training run.
for name in ('resnet18', 'vit'):
    flops = pk.count_flops_per_image(lambda name=name: M.build(name, 1000))   # 1000 classes is the ImageNet head.
    work  = pk.workload_spec(name, n_train=IN32_N, n_val=50_000, batch_size=512,
                             epochs=IN32_EPOCHS, flops=flops)
    est = pk.tier1_estimate(work, p_tflops=p_tflops)        # Returns {'optimistic': {...}, 'pessimistic': {...}}.
    bench[name]['roofline_h'] = (est['optimistic']['total_s'] / 3600, est['pessimistic']['total_s'] / 3600)
    print(f'{name:9s}: roofline predict ImageNet-32 = '
          f'{bench[name]["roofline_h"][0]:.2f} – {bench[name]["roofline_h"][1]:.2f} h   '
          f'(naive was {bench[name]["naive_hours"]:.2f} h)')
print('\nTwo predictions on the table. §9 runs it and checks who was right.')

## 9. ImageNet-32 — where the CNN breaks down, and predicted vs **actual**

Now the payoff, and the reason for every optimization above. We train the *same two models* on
**ImageNet-32** — 1,281,167 images, 1000 classes, still 32×32 — using the GPU-resident loader from
`imagenet_data.py` (the whole 3.9 GB dataset lives in VRAM; there is no DataLoader).

Two things to watch:

- **The crossover.** On CIFAR's 50k the CNN won easily. Here, with **25× the data**, the ViT is no longer
  starved — it *learns* the spatial structure the CNN gets for free, and closes or beats the gap. In the
  full study the from-scratch **ViT overtakes** (val top-1 **42.76%** vs the ResNet's **41.54%**). That is
  the whole thesis: *CNNs win on small data, Transformers win once the data is there.* Why not always use
  the faster-to-train CNN? Because its inductive bias is also a **ceiling** — the ViT keeps improving with
  data past where the CNN plateaus.
- **Predicted vs actual.** We timed §8's predictions; now we measure the real thing and see how close we
  got — and what to fix.

> **`MODE='fast'` runs a ~30k-image subset for a couple of epochs** — enough to prove the pipeline and the
> prediction *method*, **not** enough to show the real crossover (that needs the full 1.28M, i.e.
> `MODE='full'`, hours). The notebook says so honestly below.

In [ ]:
from imagenet_data import GpuImageNet32                 # The GPU-resident ImageNet-32 loader; see imagenet_data.py.

# Step 1: Load ImageNet-32 into VRAM. In 'full', subset=None pulls all 1,281,167 images (about 3.9 GB
# uint8) onto the card at once, the same "hold it all on the GPU" trick as CIFAR in section 3, just
# bigger. The .npy is sorted by class, so a sequential slice would grab a handful of classes and lie to
# you about accuracy. 'fast' passes a subset size and data.py takes a random sample across all 1000.
in32_train = GpuImageNet32(DEVICE, split='train', subset=CONFIG['in32_subset'], seed=0)
in32_val   = GpuImageNet32(DEVICE, split='val',   subset=None)
print(f"ImageNet-32 resident on {DEVICE}: {in32_train.gb():.1f} GB  |  "
      f"{in32_train.n:,} train, {in32_val.n:,} val, {in32_train.n_classes} classes")

# Step 2: Adapt the loader to train()'s contract. GpuImageNet32.epoch(bs, train) yields augmented
# (train) or clean (val) (x, y) batches, exactly what train_batches()/val_batches() must return, so a
# one-line lambda is the whole adapter. This is the same train() from section 6; nothing about the loop
# changes between 50k CIFAR images and 1.28M ImageNet images.
BS = 512
in32_batches     = lambda: in32_train.epoch(BS, train=True)
in32_val_batches = lambda: in32_val.epoch(BS, train=False)

In [ ]:
# Step 1: Train ResNet-18 on ImageNet-32, and time the whole run: we hold this actual wall-clock up
# against section 8's two predictions below. num_classes comes from the dataset (1000), never a hardcode.
t0 = time.time()
in32_resnet = train('ResNet-18 / ImageNet-32',
                    M.build('resnet18', num_classes=in32_train.n_classes),
                    in32_batches, in32_val_batches,
                    n_train=in32_train.n, epochs=IN32_EPOCHS, lr=0.1 * 512 / 256)   # Same batch-scaled SGD LR as CIFAR.
resnet_actual_h = (time.time() - t0) / 3600
print(f'ResNet-18 ImageNet-32 best val top-1: {max(in32_resnet["top1"]):.2%}   '
      f'(actual wall-clock {resnet_actual_h*60:.1f} min)')

In [ ]:
# Step 2: The same for the ViT, with the transformer recipe (AdamW plus strong_aug), timed the same
# way. On this much data the strong augmentation is earning its keep: recall from imagenet_data.py that without
# it the ViT memorized the training set (97% train / 33% val, a 65-point gap).
t0 = time.time()
in32_vit = train('ViT / ImageNet-32',
                 M.build('vit', num_classes=in32_train.n_classes),
                 in32_batches, in32_val_batches,
                 n_train=in32_train.n, epochs=IN32_EPOCHS, lr=1e-3, opt='adamw', strong_aug=vit_aug)
vit_actual_h = (time.time() - t0) / 3600
print(f'ViT ImageNet-32 best val top-1: {max(in32_vit["top1"]):.2%}   '
      f'(actual wall-clock {vit_actual_h*60:.1f} min)')

In [ ]:
# Step 3: The whole thesis in one figure, the same two models: left on 50k images, right on 1.28M.
# What to look for: on the left the CNN curve sits above the ViT's (its locality prior wins on little
# data); on the right the gap closes or flips (given enough data the ViT learns the structure the CNN
# hard-codes, then keeps going past where the CNN plateaus).
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(range(1, len(resnet_hist['top1'])+1), resnet_hist['top1'], color='#2a6', label='ResNet-18')
a1.plot(range(1, len(vit_hist['top1'])+1),    vit_hist['top1'],    color='#96c', label='ViT')
a1.set_title(f'CIFAR-10 ({c_n:,} imgs) — CNN ahead'); a1.set_xlabel('epoch'); a1.set_ylim(0,1)
a1.legend(); a1.grid(alpha=.3)
a2.plot(range(1, len(in32_resnet['top1'])+1), in32_resnet['top1'], color='#2a6', label='ResNet-18')
a2.plot(range(1, len(in32_vit['top1'])+1),    in32_vit['top1'],    color='#96c', label='ViT')
a2.set_title(f'ImageNet-32 ({in32_train.n:,} imgs) — the gap closes'); a2.set_xlabel('epoch'); a2.set_ylim(0,1)
a2.legend(); a2.grid(alpha=.3); plt.tight_layout(); plt.show()
# In 'fast' MODE these are a subset for a couple of epochs, so the shape is a preview, not the verdict.
if MODE == 'fast':
    print("NOTE: MODE='fast' — subset/short. The real crossover (ViT 42.76% > CNN 41.54%) needs MODE='full'.")

In [ ]:
# Step 1: Put the actual on the same footing as the predictions. In 'fast' MODE we only trained a
# subset for a few epochs, but section 8 predicted the full 1.28M by 40-epoch run, so we scale the
# measured time up by (full images*epochs / what we actually ran). It is a linear scale-up, fair
# because the inner training loop dominates the wall-clock. (In 'full' MODE this factor is about 1, so
# it is a no-op.)
def to_full_hours(actual_h):
    ran = in32_train.n * IN32_EPOCHS                        # image*epochs we actually trained.
    full = 1_281_167 * IN32_EPOCHS                          # image*epochs the section 8 predictions assumed.
    return actual_h * full / ran

# Step 2: Line up naive versus roofline versus the measured truth, and print how far off the naive
# number was.
print(f"{'model':9s} {'naive':>8s} {'roofline':>16s} {'ACTUAL':>10s}   (hours, 40-epoch full run)")
for name, actual_h in [('resnet18', resnet_actual_h), ('vit', vit_actual_h)]:
    b = bench[name]; act = to_full_hours(actual_h)
    lo, hi = b['roofline_h']
    err = (b['naive_hours'] - act) / act * 100
    print(f"{name:9s} {b['naive_hours']:7.2f}h {lo:6.2f}-{hi:5.2f}h {act:9.2f}h   naive off {err:+.0f}%")
print("""
Reading it:  the NAIVE img/s extrapolation tends to UNDER-predict, and the fixes are the lesson —
  * ImageNet has a 1000-class head (CIFAR's was 10): more FLOPs per step the CIFAR benchmark never saw.
    perfkit's roofline counts them, which is why its band brackets the truth better.
  * per-epoch eval on 50k val images, and epoch-1 cuDNN autotune, are real wall-clock the naive
    images*epochs/img_s ignores.
  * the ViT's strong augmentation (mixup/CutMix/erasing) costs per-batch time the CIFAR CNN benchmark
    didn't include.
The fix is the factory principle: **benchmark the SAME model + head + augmentation you'll run**, or use a
FLOPs-aware estimate — then the prediction lands, and you can schedule a day of runs with confidence.""")

## 10. The model factory

Stack the pieces and you have the backbone `hyper_sweep.py` was reaching for:

- **runs anywhere** — one `get_device()`, four platforms (§2), debugged through the MPS/spawn/`.view()`
  scars (§3–4);
- **fast** — on-device data, AMP, the measured `channels_last`-by-dtype rule, the bound-diagnosis habit (§3–5);
- **both families, at any scale** — CNN *and* Transformer, from 50k to 1.28M images, on one loop (§6–9);
- **predictable** — cost estimated before the run, then checked against reality and corrected (§8–9).

A **model factory** points that at a search space — architecture family, width/depth, recipe — and lets it
*estimate → schedule → run* across whatever hardware is on hand (the two RTX PRO 6000s here, a Mac
overnight, a Colab tab), keeping the accuracy/cost Pareto front, and choosing **CNN or Transformer by the
data budget** it's given. The **what to search** is the crossover we just watched; the **how to run it
cheaply, everywhere, and know the bill in advance** is everything above.

We started at `sklearn.fit()` and a hand-written `Solver`. This is where that goes.